
# CSV Replicator: 10×, 100×, 1000×

This notebook takes an input CSV and **replicates its rows** 10x, 100x, and 1000x, 
saving the replicated datasets as separate CSV files **in the same folder** as the input.

**How to use:**
1. Put this notebook in the same folder as your CSV (or update `input_csv` with the full path).
2. Run the cells below.  
3. You will get files named like: `yourfile_x10.csv`, `yourfile_x100.csv`, `yourfile_x1000.csv`.

**Notes:**
- The replication is simply repeating the rows; no shuffling or changes to content.
- Index is not saved to the output CSVs.
- Uses pandas; install with `pip install pandas` if needed.


In [3]:

# --- Configuration ---
# Set the path to your input CSV here. If your CSV is in the same directory, just put the filename.
input_csv = "airline.csv"  # <-- change this to your CSV filename

# Replication factors you want to output
factors = [10, 100, 1000]


In [4]:

import pandas as pd
from pathlib import Path

def replicate_csv(input_path: str | Path, k: int, output_path: str | Path | None = None) -> Path:
    """
    Load a CSV once and replicate its rows k times, writing to output_path.
    Returns the Path to the created file.
    """
    input_path = Path(input_path)
    if output_path is None:
        stem = input_path.stem
        suffix = input_path.suffix or ".csv"
        output_path = input_path.with_name(f"{stem}_x{k}{suffix}")
    else:
        output_path = Path(output_path)

    # Read once
    df = pd.read_csv(input_path)

    # Simple replication: concatenate the same DataFrame k times
    # This preserves column order and dtypes as read by pandas.
    replicated = pd.concat([df] * k, ignore_index=True)

    # Save without index
    replicated.to_csv(output_path, index=False)
    return output_path


In [ ]:

# --- Run: create 10x, 100x, 1000x outputs ---
in_path = Path(input_csv)
assert in_path.exists(), f"Input CSV not found: {in_path.resolve()}"

created = []
for k in factors:
    out_path = replicate_csv(in_path, k)
    created.append(out_path)

print("Created files:")
for p in created:
    print("-", p)



## Optional: Memory-friendly approach (very large files)

If your CSV is huge and `concat` causes memory issues, consider this pattern instead:

1. Stream the input file in chunks.
2. Write each chunk repeatedly to the output file in append mode.

Replace the function above with this chunked version if needed.


In [ ]:

import pandas as pd
from pathlib import Path

def replicate_csv_chunked(input_path: str | Path, k: int, output_path: str | Path | None = None, chunksize: int = 100_000) -> Path:
    input_path = Path(input_path)
    if output_path is None:
        stem = input_path.stem
        suffix = input_path.suffix or ".csv"
        output_path = input_path.with_name(f"{stem}_x{k}{suffix}")
    else:
        output_path = Path(output_path)

    # Remove existing output if present
    if output_path.exists():
        output_path.unlink()

    header_written = False
    for chunk in pd.read_csv(input_path, chunksize=chunksize):
        # Write this chunk k times
        for i in range(k):
            chunk.to_csv(output_path, mode="a", index=False, header=not header_written)
            header_written = True
    return output_path
